In [1]:
import numpy as np

# -------------------- Data --------------------
x = np.array([[0,0],
              [0,1],
              [1,0],
              [1,1]])

y = np.array([[1,0],
              [0,1],
              [0,1],
              [1,0]])

np.random.seed(1)

# -------------------- Activations --------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def dsigmoid(a):
    return a * (1 - a)

def tanh(z):
    return np.tanh(z)

def dtanh(a):
    return 1 - a**2

def relu(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def swish(z):
    return z * sigmoid(z)

def dswish(z):
    sig = sigmoid(z)
    return sig + z * sig * (1 - sig)

# -------------------- Training Function --------------------
def train(hidden_act, hidden_dact, output_act,
          epochs=500, lr=0.59, use_softmax=False):

    # Initialize weights
    w1 = np.random.randn(2, 8)
    b1 = np.zeros((1, 8))
    w2 = np.random.randn(8, 6)
    b2 = np.zeros((1, 6))
    w3 = np.random.randn(6, 4)
    b3 = np.zeros((1, 4))
    w4 = np.random.randn(4, 2)
    b4 = np.zeros((1, 2))

    for epoch in range(epochs):

        # -------- Forward pass --------
        z1 = x @ w1 + b1
        a1 = hidden_act(z1)

        z2 = a1 @ w2 + b2
        a2 = hidden_act(z2)

        z3 = a2 @ w3 + b3
        a3 = hidden_act(z3)

        z4 = a3 @ w4 + b4
        output = output_act(z4)

        # -------- Loss --------
        if use_softmax:
            loss = -np.mean(np.sum(y * np.log(output + 1e-9), axis=1))
            d_output = output - y          # softmax + CE gradient
        else:
            loss = np.mean((output - y) ** 2)
            d_output = (output - y) * dsigmoid(output)

        # -------- Backprop --------
        dw4 = a3.T @ d_output
        db4 = np.sum(d_output, axis=0, keepdims=True)

        da3 = d_output @ w4.T
        dz3 = da3 * hidden_dact(a3)

        dw3 = a2.T @ dz3
        db3 = np.sum(dz3, axis=0, keepdims=True)

        da2 = dz3 @ w3.T
        dz2 = da2 * hidden_dact(a2)

        dw2 = a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        da1 = dz2 @ w2.T
        dz1 = da1 * hidden_dact(a1)

        dw1 = x.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        # -------- Update --------
        w4 -= lr * dw4; b4 -= lr * db4
        w3 -= lr * dw3; b3 -= lr * db3
        w2 -= lr * dw2; b2 -= lr * db2
        w1 -= lr * dw1; b1 -= lr * db1

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    print("Predictions:\n", output)
    print("-" * 40)

# -------------------- Experiments --------------------
print("Using Sigmoid")
train(sigmoid, dsigmoid, sigmoid)

print("Using Tanh")
train(tanh, dtanh, sigmoid)

print("Using ReLU")
train(relu, drelu, sigmoid)

print("Using Swish")
train(swish, dswish, sigmoid)

print("Using Softmax + Cross Entropy")
train(relu, drelu, softmax, use_softmax=True)


Using Sigmoid
Epoch 0, Loss: 0.2874
Epoch 100, Loss: 0.2494
Epoch 200, Loss: 0.2463
Epoch 300, Loss: 0.2268
Epoch 400, Loss: 0.0681
Predictions:
 [[0.89923977 0.09128834]
 [0.09881988 0.90837799]
 [0.104405   0.90402486]
 [0.91723832 0.07700051]]
----------------------------------------
Using Tanh
Epoch 0, Loss: 0.2457
Epoch 100, Loss: 0.0026
Epoch 200, Loss: 0.0010
Epoch 300, Loss: 0.0006
Epoch 400, Loss: 0.0005
Predictions:
 [[0.9852856  0.01533173]
 [0.02338155 0.97582165]
 [0.01951393 0.97909039]
 [0.98565834 0.0146701 ]]
----------------------------------------
Using ReLU
Epoch 0, Loss: 0.2852
Epoch 100, Loss: 0.0034
Epoch 200, Loss: 0.0014
Epoch 300, Loss: 0.0009
Epoch 400, Loss: 0.0007
Predictions:
 [[0.96815989 0.03164179]
 [0.00463317 0.99675502]
 [0.00380708 0.99623604]
 [0.96815989 0.03164179]]
----------------------------------------
Using Swish
Epoch 0, Loss: 0.3381
Epoch 100, Loss: 0.0012
Epoch 200, Loss: 0.0000
Epoch 300, Loss: 0.0000
Epoch 400, Loss: 0.0000
Predictions: